# 生成式AI應用領域與工具使用

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明生成式 AI 與傳統辨識型 AI 的差異。
2. 了解生成式 AI 的常見技術概念：標記化、向量化、推理取樣、溫度參數、Top-k 與 Nucleus Sampling。
3. 使用輕量 Python 方法模擬生成式 AI 工具的核心流程。
4. 分析生成式 AI 在教育、醫療、金融、創意產業等領域的應用。
5. 從市場價值、風險、工具演進與產業趨勢角度評估生成式 AI 應用。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會使用到的 Colab 預裝套件，並準備一組簡化的生成式 AI 應用資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import re

applications = pd.DataFrame({
    "領域": ["教育", "醫療", "金融", "法律", "創意產業", "軟體開發", "客服"],
    "應用案例": [
        "個人化學習路徑與教材生成",
        "臨床報告摘要與輔助診斷",
        "風險分析與投資建議生成",
        "契約草稿撰寫與法規檢索",
        "圖像、文案、音樂與影片內容創作",
        "程式碼補全、測試案例與文件產生",
        "自動回覆、知識庫問答與情緒分析"
    ],
    "主要價值": ["提升學習效率", "改善醫療流程", "降低決策成本", "提升文件處理效率", "加速創作流程", "提高開發效率", "降低客服負載"],
    "主要風險": ["內容正確性", "隱私與責任歸屬", "模型偏誤", "法規遵循", "著作權", "程式安全", "錯誤回覆"]
})

print(applications)


## 核心概念說明

生成式 AI 的重點不是只判斷資料屬於哪一類，而是根據訓練資料的模式產生新的文字、圖像、音訊、程式碼或其他內容。

在大型語言模型中，常見流程可簡化為：

1. **資料處理**：清洗文字、移除雜訊、切分成 token。
2. **向量化表示**：將文字轉成數值向量，方便模型計算相似度或語意關係。
3. **模型推理**：根據目前上下文，估計下一個 token 或輸出內容。
4. **取樣策略**：使用溫度參數、Top-k 或 Nucleus Sampling 控制輸出的保守或創意程度。
5. **應用整合**：將模型能力放進教育、醫療、客服、開發、創作等工作流程。

本 Notebook 不使用大型模型，而是用 TF-IDF、機率分布與簡化取樣來理解生成式 AI 的核心概念。


In [ ]:
# ── 示範：標記化與文字向量化 ────────────────────────────
# 這段程式碼用正規表示式進行簡易標記化，並用 TF-IDF 模擬文字向量化，對應教材中的 Tokenization 與 Vectorization。

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [
    "生成式AI可以產生教材、摘要與練習題",
    "醫療AI可以協助產生臨床報告摘要",
    "金融AI可以進行風險分析與投資建議",
    "創意產業使用AI產生圖像、文案與影片腳本"
]

def tokenize_zh_like(text):
    return re.findall(r"[\u4e00-\u9fffA-Za-z0-9]+", text)

for i, text in enumerate(texts, 1):
    print(f"文本 {i} tokens:", tokenize_zh_like(text))

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf = vectorizer.fit_transform(texts)

result = pd.DataFrame(tfidf.toarray(), columns=vectorizer.get_feature_names_out())
print("\nTF-IDF 向量表：")
print(result.round(3))


## 推理機制：溫度、Top-k 與 Nucleus Sampling

生成式 AI 產生內容時，通常不是永遠選擇機率最高的答案。若永遠選最大值，輸出會穩定但可能缺乏變化；若加入取樣，輸出會更有多樣性。

常見控制方式包括：

- **Temperature**：控制隨機性。數值低時較保守，數值高時較有創意。
- **Top-k Sampling**：只從機率最高的前 k 個候選中取樣。
- **Nucleus Sampling**：只保留累積機率達到門檻 p 的候選，例如 0.9。

這些方法常用於文字生成、摘要、對話、程式碼生成與創意內容產生。


In [ ]:
# ── 示範：溫度參數與 Top-k 取樣 ───────────────────────
# 這段程式碼用一組候選詞的機率分布，模擬生成式 AI 在不同溫度與 Top-k 設定下的輸出差異。

import numpy as np
import pandas as pd

np.random.seed(7)

candidates = np.array(["教材", "摘要", "圖像", "程式碼", "報告", "影片", "音樂"])
base_logits = np.array([3.0, 2.5, 2.1, 1.8, 1.4, 1.0, 0.7])

def softmax(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum()

def sample_with_temperature_and_topk(logits, temperature=1.0, top_k=3, n=10):
    adjusted = logits / temperature
    probs = softmax(adjusted)
    top_indices = np.argsort(probs)[-top_k:][::-1]
    top_probs = probs[top_indices]
    top_probs = top_probs / top_probs.sum()
    sampled = np.random.choice(candidates[top_indices], size=n, p=top_probs)
    return sampled, pd.DataFrame({
        "候選詞": candidates[top_indices],
        "重新正規化後機率": top_probs
    })

for temp in [0.5, 1.0, 1.8]:
    sampled, table = sample_with_temperature_and_topk(base_logits, temperature=temp, top_k=4, n=8)
    print(f"\nTemperature = {temp}")
    print(table.round(3))
    print("生成結果：", "、".join(sampled))


In [ ]:
# ── 實際應用：生成式 AI 工具推薦器 ───────────────────────
# 這段程式碼使用 TF-IDF 與餘弦相似度，模擬一個簡化的 AI 工具推薦器；使用者輸入需求後，系統會推薦最相關的應用領域。

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cases = pd.DataFrame({
    "工具類型": ["教育生成工具", "醫療摘要工具", "金融分析工具", "法律文件工具", "創意內容工具", "程式開發工具", "客服問答工具"],
    "描述": [
        "教材 學習 測驗 課程摘要 練習題 個人化",
        "臨床報告摘要 醫療影像說明 診斷 醫療",
        "投資建議 風險分析 市場報告 金融",
        "契約草稿 法規檢索 文件審閱 法律",
        "文案 圖像 腳本 音樂 創意內容",
        "程式碼 測試案例 技術文件 除錯建議",
        "客服回覆 知識庫問答 使用者情緒分析"
    ]
})

user_need = "課程摘要 學生練習題 教材 學習"

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
all_texts = list(cases["描述"]) + [user_need]
vectors = vectorizer.fit_transform(all_texts)

similarities = cosine_similarity(vectors[-1], vectors[:-1]).flatten()
cases["相似度"] = similarities
recommendation = cases.sort_values("相似度", ascending=False)

print("使用者需求：", user_need)
print("\n推薦結果：")
print(recommendation[["工具類型", "相似度"]].round(3))


In [ ]:
# ── 實際應用：市場價值與風險視覺化 ─────────────────────────
# 這段程式碼建立簡化的市場影響評估表，並用長條圖比較不同領域的預估價值與風險分數。

import pandas as pd
import matplotlib.pyplot as plt

market = pd.DataFrame({
    "領域": ["教育", "醫療", "金融", "法律", "創意產業", "軟體開發", "客服"],
    "價值分數": [8, 9, 8, 7, 9, 9, 8],
    "風險分數": [5, 9, 8, 8, 7, 6, 6]
})

market["淨機會分數"] = market["價值分數"] - market["風險分數"]
print(market.sort_values("淨機會分數", ascending=False))

plt.figure(figsize=(9, 4))
plt.bar(market["領域"], market["價值分數"], label="價值分數")
plt.bar(market["領域"], -market["風險分數"], label="風險分數")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("生成式 AI 應用領域的價值與風險比較")
plt.xlabel("應用領域")
plt.ylabel("分數")
plt.legend()
plt.tight_layout()
plt.show()
